#Explorando Dados na Web com Python

In [3]:
import json
import requests
import pandas as pd

from urllib.parse import parse_qs, urlencode, urlparse

##Nível 1: Básico (URLs, Parâmetros e Cabeçalhos)

Exercício 1.1: Faça uma requisição GET para a API pública do JSONPlaceholder ([https://jsonplaceholder.typicode.com/posts](https://jsonplaceholder.typicode.com/posts)) ou para a API do IBGE.

In [4]:
try:
    resposta = requests.get('https://jsonplaceholder.typicode.com/posts', timeout=10)
    print(resposta.text)
except requests.RequestException as erro:
    print('SEM internet:', erro)

[
  {
    "userId": 1,
    "id": 1,
    "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
    "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
  },
  {
    "userId": 1,
    "id": 2,
    "title": "qui est esse",
    "body": "est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla"
  },
  {
    "userId": 1,
    "id": 3,
    "title": "ea molestias quasi exercitationem repellat qui ipsa sit aut",
    "body": "et iusto sed quo iure\nvoluptatem occaecati omnis eligendi aut ad\nvoluptatem doloribus vel accusantium quis pariatur\nmolestiae porro eius odio et labore et velit aut"
  },
  {
    "userId": 1,
    "id": 4,
    "title": "eum et est occaecati",
    "body": "ullam et saepe reic

Exercício 1.2: Utilize o argumento params para filtrar os resultados (por exemplo, buscando apenas o userId igual a 2 ou limitando a quantidade de resultados).

In [5]:
filtros = {'userId': 2, '_limit': 2}
resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params=filtros,
    timeout=10
)

resposta.json()

[{'userId': 2,
  'id': 11,
  'title': 'et ea vero quia laudantium autem',
  'body': 'delectus reiciendis molestiae occaecati non minima eveniet qui voluptatibus\naccusamus in eum beatae sit\nvel qui neque voluptates ut commodi qui incidunt\nut animi commodi'},
 {'userId': 2,
  'id': 12,
  'title': 'in quibusdam tempore odit est dolorem',
  'body': 'itaque id aut magnam\npraesentium quia et ea odit et ea voluptas et\nsapiente quia nihil amet occaecati quia id voluptatem\nincidunt ea est distinctio odio'}]

Exercício 1.3: Na sua requisição, inclua um dicionário de cabeçalhos (headers) passando um User-Agent personalizado com o nome do seu projeto.

In [6]:
resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params={'userId': 2, '_limit': 2},
    headers={
        'User-Agent': 'Explorando Dados na Web com Python'
    },
    timeout=10
)

resposta.json()

[{'userId': 2,
  'id': 11,
  'title': 'et ea vero quia laudantium autem',
  'body': 'delectus reiciendis molestiae occaecati non minima eveniet qui voluptatibus\naccusamus in eum beatae sit\nvel qui neque voluptates ut commodi qui incidunt\nut animi commodi'},
 {'userId': 2,
  'id': 12,
  'title': 'in quibusdam tempore odit est dolorem',
  'body': 'itaque id aut magnam\npraesentium quia et ea odit et ea voluptas et\nsapiente quia nihil amet occaecati quia id voluptatem\nincidunt ea est distinctio odio'}]

Exercício 1.4: Imprima na tela o código de status HTTP (status_code) para confirmar o sucesso (código 200) e a URL final montada pelo requests (resposta.url). Lembre-se de sempre utilizar o parâmetro timeout (ex: timeout=10) para evitar que seu código fique travado indefinidamente.

In [7]:
print(resposta.status_code)
print(resposta.url)

200
https://jsonplaceholder.typicode.com/posts?userId=2&_limit=2


##Nível 2: Intermediário (JSON, Erros e Arquivos Binários)

Exercício 2.1: Crie uma lista com três CEPs diferentes e faça um loop para consultá-los na API do ViaCEP ([https://viacep.com.br/ws/](https://viacep.com.br/ws/){cep}/json/). Utilize o método .json() para converter as respostas e armazene os resultados em um DataFrame do Pandas.

In [9]:
cep = ['24921744', '24909305', '24922085']
registros = []

for c in cep:
    resposta = requests.get(f'https://viacep.com.br/ws/{c}/json/', timeout=10)
    if resposta.ok:
      registros.append(resposta.json())

df_cep = pd.DataFrame(registros)
df_cep

,cep,logradouro,complemento,unidade,bairro,localidade,uf,estado,regiao,ibge,gia,ddd,siafi
0,24921-744,Rua São João del Rei,,,Cordeirinho (Ponta Negra),Maricá,RJ,Rio de Janeiro,Sudeste,3302700,,21,5853
1,24909-305,Rua dos Canários,(Cond Res G Éden),,Pilar,Maricá,RJ,Rio de Janeiro,Sudeste,3302700,,21,5853
2,24922-085,Rua Cento e Quarenta e Nove,,,Ponta Negra (Ponta Negra),Maricá,RJ,Rio de Janeiro,Sudeste,3302700,,21,5853


Exercício 2.2: Escreva uma função de download segura utilizando um bloco try/except. Dentro dela, utilize resposta.raise_for_status() para capturar erros HTTP (como 404 ou 500) e imprima uma mensagem amigável caso a requisição falhe.

In [15]:
def baixar_arquivo(url, nome_arquivo):
    try:
        resposta = requests.get(url, timeout=10)
        resposta.raise_for_status()
        with open(nome_arquivo, 'wb') as arquivo:
            arquivo.write(resposta.content)
        return resposta
    except requests.RequestException as erro:
        print(f'Erro ao acessar o arquivo: {url}\nError: {erro}')

Exercício 2.3: Faça uma requisição para a URL [https://picsum.photos/400/400](https://picsum.photos/400/400) para baixar uma imagem aleatória. Salve o conteúdo bruto da resposta (acessado através de resposta.content) em um arquivo local com a extensão .jpg utilizando o modo de escrita em bytes ('wb').

In [16]:
resposta = baixar_arquivo('https://picsum.photos/400/400', 'imagem.jpg')
with open('imagem.jpg', 'wb') as arquivo:
    arquivo.write(resposta.content)

##Nível 3: Avançado (Webscraping e Ética)

Exercício 3.1: Antes de fazer a raspagem, faça uma requisição para o arquivo robots.txt do site que deseja acessar para verificar as permissões de acesso, demonstrando responsabilidade ética na coleta de dados.

In [24]:
robots = requests.get('https://books.toscrape.com/robots.txt', headers={'User-Agent': 'aula_web'}, timeout=10)
print(robots.text)

<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.21.6</center>
</body>
</html>



Exercício 3.2: Acesse o site voltado para estudos de raspagem [https://books.toscrape.com/](https://books.toscrape.com/). Utilize a biblioteca BeautifulSoup com o analisador 'html.parser' para localizar os elementos da página. Extraia o título e o preço dos 5 primeiros livros utilizando métodos como .find() ou .select().

In [19]:
from bs4 import BeautifulSoup

In [26]:
resposta = requests.get('https://books.toscrape.com/', headers={'User-Agent': 'aula_web'}, timeout=10)
resposta.raise_for_status()

soup = BeautifulSoup(resposta.text, 'html.parser')
livros = soup.select('article.product_pod')

dados = []
for livro in livros[:5]:
  dados.append({
        'titulo': livro.h3.a['title'],
        'preco': livro.select_one('p.price_color').text
  })

df_livros = pd.DataFrame(dados)
df_livros


,titulo,preco
0,A Light in the Attic,Â£51.77
1,Tipping the Velvet,Â£53.74
2,Soumission,Â£50.10
3,Sharp Objects,Â£47.82
4,Sapiens: A Brief History of Humankind,Â£54.23


Exercício 3.3: Salve os dados extraídos dos livros em um arquivo CSV utilizando o Pandas.

In [27]:
df_livros.to_csv('livros.csv', index=False, sep=';')
print('Arquivo salvo com sucesso!')
print(df_livros.to_string())

Arquivo salvo com sucesso!
                                  titulo    preco
0                   A Light in the Attic  Â£51.77
1                     Tipping the Velvet  Â£53.74
2                             Soumission  Â£50.10
3                          Sharp Objects  Â£47.82
4  Sapiens: A Brief History of Humankind  Â£54.23


Exercício 3.4: Escolha uma página da Wikipedia que contenha uma tabela de dados (como listas de países ou populações). Utilize a função pandas.read_html() combinada com io.StringIO() para capturar a tabela da página e transformá-la diretamente em um DataFrame, sem a necessidade de usar o BeautifulSoup.

In [29]:
import io

pagina = requests.get('https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o',
                      headers={'User-Agent': 'aula_web'},
                      timeout=10
         )
print('Pagina acessada com sucesso!', pagina.status_code, '| bytes: ', len(pagina.content))

df_paises = pd.read_html(io.StringIO(pagina.text))[0]
df_paises

Pagina acessada com sucesso! 200 | bytes:  640735


,Unnamed: 0,Posição,País (ou território dependente),Estimativa da ONU (2021),Mudança desde a última estimativa,Estimativa Oficial
0,NaN,1,Índia,1 417 492 000,-12 692 000,Estimativa oficial
1,NaN,2,China[nota 1],1 407 934 000,-1 390 000,Censo oficial
2,NaN,3,Estados Unidos,342 181 000,-2 084 000,Censo oficial
3,NaN,4,Indonésia,285 783 000,+3 337 000,Estimativa oficial
4,NaN,5,Paquistão,256 204 000,+6 567 000,Estimativa oficial
...,...,...,...,...,...,...
245,NaN,–,Ilha de Ascensão (Reino Unido),1 100,NaN,Estimativa oficial[10]
246,NaN,195,Vaticano,879,NaN,Estimativa oficial
247,NaN,–,[[File:|22x20px|border |alt=|link=]] Ilhas Coc...,605,NaN,Estimativa oficial
248,NaN,–,Tristão da Cunha (Reino Unido),264,NaN,Estimativa oficial[11]
